In [1]:
import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")

import pessoal
from pessoal import *

spark.sparkContext.setLogLevel("ERROR")

Tempo inicial da execucao: 2025-11-22 15:47:52.470315
User: tatiane
Node: tatiane-Inspiron-3583


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/22 15:47:56 WARN Utils: Your hostname, tatiane-Inspiron-3583, resolves to a loopback address: 127.0.1.1; using 192.168.0.14 instead (on interface wlo1)
25/11/22 15:47:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/22 15:47:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/22 15:48:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Geração automaticamente do schema Spark

In [2]:
import re
from pyspark.sql.types import StructType, StructField, StringType, FloatType

LAYOUT_FILE = "/home/tatiane/Downloads/NINSOC/Dicionario_e_input_20221031/input_PNADC_trimestral.txt"

def load_layout(layout_file=LAYOUT_FILE):
    """
    Lê o arquivo SAS de layout da PNADC e retorna uma lista de:
    (start_position, variable_name, spark_type, width)
    """
    pattern = r"@(\d+)\s+(\w+)\s+(\$?)(\d+)\."
    fields = []

    with open(layout_file, "r", encoding="latin1") as f:
        lines = f.readlines()

    for line in lines:
        m = re.search(pattern, line)
        if m:
            start = int(m.group(1))
            name = m.group(2)
            is_string = m.group(3) == "$"
            width = int(m.group(4))
            spark_type = StringType() if is_string else FloatType()
            fields.append((start, name, spark_type, width))

    # Ordenar por posição
    fields.sort(key=lambda x: x[0])
    return fields

In [3]:
from pyspark.sql.functions import substring, col, trim, when

def read_pnadc(txt_file_path, layout_file=LAYOUT_FILE):
    """
    Lê um arquivo TXT de microdados PNADC (qualquer ano) usando o layout especificado.
    """
    # Carregar metadados do layout
    fields = load_layout(layout_file)

    # Ler o arquivo como texto puro
    raw = spark.read.text(txt_file_path)

    # Criar colunas baseadas em posições fixas
    df = raw
    for start, name, spark_type, width in fields:
        df = df.withColumn(name, substring("value", start, width))

    df = df.drop("value")

    # Converter espaços em branco para NULL
    df = df.select([
        when(trim(col(c)) == "", None).otherwise(col(c)).alias(c)
        for c in df.columns
    ])

    # Aplicar cast final
    for _, name, spark_type, _ in fields:
        df = df.withColumn(name, col(name).cast(spark_type))

    return df

#### Leitura das bases e seleção das variáveis

In [4]:
df1 = read_pnadc("/home/tatiane/Downloads/NINSOC/PNADC_012023.txt")
df2 = read_pnadc("/home/tatiane/Downloads/NINSOC/PNADC_022023.txt")
df3 = read_pnadc("/home/tatiane/Downloads/NINSOC/PNADC_032023.txt")
df4 = read_pnadc("/home/tatiane/Downloads/NINSOC/PNADC_042023.txt")

In [5]:
select_var = ['Ano', 'V2007', 'V2010', 'V2009', 'V2001', 'V2005', 'V1022', 'UF', 'VD4019', 'V1027', 'VD4015'] 

In [7]:
df1_sel = df1.select(select_var)
df2_sel = df2.select(select_var)
df3_sel = df3.select(select_var)
df4_sel = df4.select(select_var)

In [9]:
df1_sel.write.mode("overwrite").parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_01")
df2_sel.write.mode("overwrite").parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_02")
df3_sel.write.mode("overwrite").parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_03")
df4_sel.write.mode("overwrite").parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_04")

### Leitura das bases parquet

In [2]:
pt_2023 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_01")
st_2023 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_02")
tt_2023 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_03")
qt_2023 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/parquet/2023_04")

In [3]:
print("Primeiro trimeste de 2023: ")
pessoal.completudeSchema(pt_2023)

print("Segundo trimeste de 2023: ")
pessoal.completudeSchema(st_2023)

print("Terceiro trimeste de 2023: ")
pessoal.completudeSchema(tt_2023)

print("Quarto trimeste de 2023: ")
pessoal.completudeSchema(qt_2023)

Primeiro trimeste de 2023: 


Qtd. registros: 473335 | Quantidade de colunas:  14
root
 |-- Ano: string (nullable = true)
 |-- V2007: string (nullable = true)
 |-- V2010: string (nullable = true)
 |-- V2008: string (nullable = true)
 |-- V20081: string (nullable = true)
 |-- V20082: string (nullable = true)
 |-- V2009: float (nullable = true)
 |-- V2001: float (nullable = true)
 |-- V2005: string (nullable = true)
 |-- V1022: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- VD4019: float (nullable = true)
 |-- V1027: float (nullable = true)
 |-- VD4015: string (nullable = true)

Segundo trimeste de 2023: 
Qtd. registros: 474575 | Quantidade de colunas:  14
root
 |-- Ano: string (nullable = true)
 |-- V2007: string (nullable = true)
 |-- V2010: string (nullable = true)
 |-- V2008: string (nullable = true)
 |-- V20081: string (nullable = true)
 |-- V20082: string (nullable = true)
 |-- V2009: float (nullable = true)
 |-- V2001: float (nullable = true)
 |-- V2005: string (nullable = true)
 |-- V1022: st

### Padronização

In [4]:
# Leitura do dicionário
dicionario_path = "/home/tatiane/Downloads/NINSOC/transformation_dictionary.csv"
dic = spark.read.csv(dicionario_path, header=True, inferSchema=False)

In [5]:
import re
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ByteType, IntegerType

# 1. Função para limpar e corrigir JSON quebrado
def fix_json_string(s):
    if s is None:
        return "{}"

    s = s.strip()
    if s in ("", "{}", "{}\n"):
        return "{}"

    s = s.replace("’", "'").replace("‘", "'") # padroniza aspas horríveis ’ ‘ para '
    s = s.replace("'", '"') # troca aspas simples por duplas    
    s = re.sub(r'(\{|\s)(\d+):', r'\1"\2":', s) # corrige chaves tipo 01:, 1:, 11:
    s = re.sub(r':\s*([^",}]+)', r':"\1"', s) # adiciona aspas no valor se estiver faltando
    return s


# 2. UDF para mapear valores categóricos
@F.udf(StringType())
def map_values_udf(value, mapping_json):
    try:
        mapping = json.loads(mapping_json)
        return mapping.get(str(value), value)
    except:
        return value

# 3. Classe principal PNADFormatter
class PNADFormatter:
    def __init__(self, dic_df):
        # aplica limpeza do JSON já no carregamento
        dic_clean = dic_df.withColumn(
            "map_values_fixed",
            F.udf(fix_json_string, StringType())(F.col("map_values"))
        )
        self.dic = dic_clean.collect()

    def transform(self, df):
        out = df

        for row in self.dic:
            old = row["var"]
            new = row["new_var"]
            new_type = (row["new_type"] or "").lower()
            mapping_json = row["map_values_fixed"]

            # renomear variável
            if old in out.columns and old != new:
                out = out.withColumnRenamed(old, new)

            has_mapping = mapping_json not in ("{}", "", None)

            # Se tiver mapping → aplicar mapping e NÃO fazer cast para número
            if has_mapping:
                out = out.withColumn(
                    new,
                    map_values_udf(F.col(new), F.lit(mapping_json))
                )
                continue  # pula cast, vai para próxima variável

            # Sem mapping → variável numérica, aplica cast quando for byte/int
            if new_type == "byte":
                out = out.withColumn(new, F.col(new).cast(ByteType()))
            elif new_type == "integer":
                out = out.withColumn(new, F.col(new).cast(IntegerType()))

        return out

In [6]:
formatter = PNADFormatter(dic)
pt_2023 = formatter.transform(pt_2023)
st_2023 = formatter.transform(st_2023)
tt_2023 = formatter.transform(tt_2023)
qt_2023 = formatter.transform(qt_2023)

- Validação da padronização

In [7]:
print("Primeiro trimeste de 2023: ")
pessoal.completudeSchema(pt_2023)

print("Segundo trimeste de 2023: ")
pessoal.completudeSchema(st_2023)

print("Terceiro trimeste de 2023: ")
pessoal.completudeSchema(tt_2023)

print("Quarto trimeste de 2023: ")
pessoal.completudeSchema(qt_2023)

Primeiro trimeste de 2023: 
Qtd. registros: 473335 | Quantidade de colunas:  14
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- dia_nasc: integer (nullable = true)
 |-- mes_nasc: integer (nullable = true)
 |-- ano_nasc: integer (nullable = true)
 |-- idade_dt_referencia: integer (nullable = true)
 |-- qtd_pessoa_domicilio: integer (nullable = true)
 |-- condicao_domicilio: string (nullable = true)
 |-- situacao_domicilio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- rmh_todos_trabalhos: float (nullable = true)
 |-- peso_domicilio_pessoa: float (nullable = true)
 |-- tp_remuneracao_habitual: string (nullable = true)

Segundo trimeste de 2023: 
Qtd. registros: 474575 | Quantidade de colunas:  14
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- dia_nasc: integer (nullable = true)
 |-- mes_nasc: integer (nullable = true)
 |-- 

### Bind dos trimestres

In [8]:
df_2023 = pt_2023.unionByName(st_2023).unionByName(tt_2023).unionByName(qt_2023)

### Criação de identificador único

In [9]:
df_2023 = createId(df_2023, 'id_unico')
df_2023.select(F.min(F.col('id_unico')), F.max(F.col('id_unico'))).show()

[Stage 35:>                                                         (0 + 1) / 1]

+-------------+-------------+
|min(id_unico)|max(id_unico)|
+-------------+-------------+
|            1|      1900989|
+-------------+-------------+



### Criação de variáveis
- rendimento_habitual
- recebe_remuneracao

In [10]:
df_2023 = df_2023.withColumn(
    "rendimento_habitual",
    F.when(F.col("rmh_todos_trabalhos").isNull(), "Não aplicável")
     .otherwise("Com rendimento habitual")
)

In [11]:
df_2023 = df_2023.withColumn(
    "recebe_remuneracao",
    F.when(F.col("tp_remuneracao_habitual").isNull(), "Não recebe")
     .otherwise("Recebe")
)

In [12]:
pessoal.completudeSchema(df_2023)

Qtd. registros: 1900989 | Quantidade de colunas:  14
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- idade_dt_referencia: integer (nullable = true)
 |-- qtd_pessoa_domicilio: integer (nullable = true)
 |-- condicao_domicilio: string (nullable = true)
 |-- situacao_domicilio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- rmh_todos_trabalhos: float (nullable = true)
 |-- peso_domicilio_pessoa: float (nullable = true)
 |-- tp_remuneracao_habitual: string (nullable = true)
 |-- id_unico: long (nullable = false)
 |-- rendimento_habitual: string (nullable = false)
 |-- recebe_remuneracao: string (nullable = false)



### Escrita da base

In [14]:
df_2023.write.mode("overwrite").parquet("PNADC/ano=2023")

### Finalização do notebook

In [15]:
executionTime()

Tempo de execucao ate este ponto: 0:04:39.650016
